
# Commercial Intelligence & Profitability Analytics

## 02 — Diseño del modelo analítico comercial

**Empresa:** Distribuidora Cordillera (datos ficticios)  
**Tecnologías:** SQLite, SQL avanzado, Jupyter  
**Etapa:** Modelado y preparación de datos para Business Intelligence

### Objetivo general

Diseñar e implementar una capa analítica a partir de los datos operacionales de Distribuidora Cordillera, definiendo estructuras, relaciones, métricas y reglas de negocio que permitan construir indicadores comerciales confiables.

### Alcance

- Identificación de procesos y entidades relevantes.
- Definición de granularidad y relaciones.
- Diseño de tablas de hechos y dimensiones.
- Construcción de vistas SQL reutilizables.
- Definición y validación de indicadores comerciales.
- Preparación de fuentes para Excel y Power BI.

### Consideraciones metodológicas

El diseño utilizará los resultados obtenidos en el notebook `01_auditoria_datos.ipynb`.

Las decisiones de modelado deberán preservar la granularidad de los registros, evitar duplicaciones en los cálculos y distinguir entre importes de pedidos entregados, pendientes y cancelados.

El proyecto utiliza datos ficticios y tiene fines de demostración de competencias en análisis de datos y Business Intelligence.


In [45]:
%load_ext sql
%sql sqlite:///../data/distribuidora_cordillera.sqlite

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Switching to connection 'sqlite:///../data/distribuidora_cordillera.sqlite'


## 2.1 Análisis conceptual del modelo de datos

### 2.1.1 Exploración de entidades y atributos

**Objetivo**

Identificar los atributos disponibles en las principales entidades comerciales, con el propósito de establecer una propuesta inicial de modelado dimensional.

**Metodología**

Se inspeccionará la estructura de las tablas `clientes`, `pedidos`, `detalle_pedidos` y `productos`, considerando sus identificadores, relaciones y atributos descriptivos o cuantitativos.

Esta exploración permitirá distinguir entre atributos de dimensión, mediciones y claves necesarias para establecer relaciones.

Las decisiones de modelado serán posteriormente contrastadas con las necesidades de análisis comercial.


In [46]:
%%sql
PRAGMA table_info(clientes);

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

cid,name,type,notnull,dflt_value,pk
0,cliente_id,INTEGER,0,None,1
1,codigo,TEXT,1,None,0
2,razon_social,TEXT,1,None,0
3,segmento,TEXT,1,None,0
4,region,TEXT,1,None,0
5,comuna,TEXT,1,None,0
6,email,TEXT,0,None,0
7,fecha_alta,TEXT,1,None,0
8,vendedor_id,INTEGER,1,None,0
9,plazo_dias,INTEGER,1,None,0


In [47]:
%%sql
PRAGMA table_info(pedidos);

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

cid,name,type,notnull,dflt_value,pk
0,pedido_id,INTEGER,0,None,1
1,cliente_id,INTEGER,1,None,0
2,vendedor_id,INTEGER,1,None,0
3,fecha_pedido,TEXT,1,None,0
4,fecha_entrega,TEXT,0,None,0
5,fecha_vencimiento,TEXT,0,None,0
6,estado,TEXT,1,None,0
7,canal,TEXT,1,None,0


In [48]:
%%sql
PRAGMA table_info(detalle_pedidos);

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

cid,name,type,notnull,dflt_value,pk
0,detalle_id,INTEGER,0,None,1
1,pedido_id,INTEGER,1,None,0
2,producto_id,INTEGER,1,None,0
3,cantidad,INTEGER,1,None,0
4,precio_unitario_clp,INTEGER,1,None,0
5,descuento_pct,INTEGER,1,None,0
6,neto_unitario_clp,INTEGER,1,None,0
7,iva_unitario_clp,INTEGER,1,None,0
8,costo_unitario_clp,INTEGER,1,None,0


In [49]:
%%sql
PRAGMA table_info(productos);

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

cid,name,type,notnull,dflt_value,pk
0,producto_id,INTEGER,0,None,1
1,sku,TEXT,1,None,0
2,nombre,TEXT,1,None,0
3,categoria_id,INTEGER,1,None,0
4,proveedor_id,INTEGER,1,None,0
5,unidad_venta,TEXT,1,None,0
6,precio_lista_clp,INTEGER,1,None,0
7,costo_referencia_clp,INTEGER,1,None,0
8,stock_minimo,INTEGER,1,None,0
9,fecha_alta,TEXT,1,None,0



### 2.1.2 Evaluación conceptual de granularidad y dimensiones

#### Granularidad propuesta

Se propone utilizar la línea de pedido como unidad de observación principal del modelo analítico comercial.

Cada registro representará una línea de producto asociada a un pedido, permitiendo calcular cantidades, importes netos, costos y márgenes brutos estimados.

Esta granularidad facilitará el análisis por producto, cliente, segmento y período.

#### Dimensiones candidatas

Se identificaron las siguientes dimensiones:

| Dimensión | Propósito |
|---|---|
| Cliente | Segmentación y distribución geográfica |
| Producto | Análisis por producto y categoría |
| Fecha | Evolución temporal y estacionalidad |
| Pedido | Características comerciales como estado y canal |

La representación definitiva de los atributos de pedido se evaluará durante el diseño dimensional, considerando la granularidad y las relaciones del modelo.

#### Consideraciones técnicas

La tabla `detalle_pedidos` contiene los identificadores `pedido_id` y `producto_id`, mientras que los atributos de cliente, fecha y estado se obtienen mediante la relación con `pedidos`.

Será necesario definir cómo incorporar estas referencias a la capa analítica.

Asimismo, se evitará repetir importes totales de pedidos en cada línea de detalle, ya que esto produciría duplicaciones al agregar los resultados.

#### Decisión preliminar

Se utilizará la granularidad de línea de pedido como punto de partida para el diseño de la tabla de hechos comercial.

Esta propuesta queda sujeta a la definición y validación de claves, dimensiones y reglas de negocio durante las siguientes etapas.



### 2.1.3 Decisiones preliminares de modelado

Se establece la línea de pedido como granularidad principal de la tabla de hechos comerciales.

Cada registro estará identificado por `detalle_id` y conservará las referencias necesarias para analizar productos, pedidos, clientes y períodos.

#### Relaciones operacionales

| Origen | Referencia | Destino | Cardinalidad |
|---|---|---|---|
| `pedidos` | `cliente_id` | `clientes.cliente_id` | N:1 |
| `detalle_pedidos` | `pedido_id` | `pedidos.pedido_id` | N:1 |
| `detalle_pedidos` | `producto_id` | `productos.producto_id` | N:1 |

#### Diseño propuesto

La tabla `FactDetallePedidos` se construirá a partir de `detalle_pedidos`, incorporando desde `pedidos` los identificadores y atributos necesarios para el análisis comercial.

Se considerarán inicialmente las dimensiones Cliente, Producto y Fecha.

Los atributos `estado` y `canal` podrán mantenerse en la tabla de hechos, sujeto a la validación del modelo.

#### Consideraciones

La repetición de claves como `cliente_id` o `pedido_id` es válida cuando varias líneas pertenecen a una misma entidad.

Sin embargo, los importes deben calcularse respetando la granularidad de cada línea para evitar duplicaciones.

La tabla de hechos distinguirá el valor registrado en los pedidos de los ingresos efectivamente realizados, cuya confirmación requiere información adicional.


In [50]:
%%sql

SELECT
    dp.detalle_id,
    p.pedido_id,
    dp.producto_id,
    p.cliente_id,
    p.fecha_pedido,
    p.estado,
    p.canal,
    dp.cantidad,
    dp.neto_unitario_clp,
    dp.costo_unitario_clp,
    dp.neto_unitario_clp * dp.cantidad AS importe_neto_linea_clp,
    dp.costo_unitario_clp * dp.cantidad AS costo_linea_clp,
    dp.neto_unitario_clp * dp.cantidad - dp.costo_unitario_clp * dp.cantidad AS margen_bruto_estimado_linea_clp
FROM detalle_pedidos AS dp
INNER JOIN pedidos AS p ON dp.pedido_id = p.pedido_id
ORDER BY dp.detalle_id ASC
;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

detalle_id,pedido_id,producto_id,cliente_id,fecha_pedido,estado,canal,cantidad,neto_unitario_clp,costo_unitario_clp,importe_neto_linea_clp,costo_linea_clp,margen_bruto_estimado_linea_clp
1,1,168,6,2024-01-01,entregado,telefono,36,4513,3400,162468,122400,40068
2,1,110,6,2024-01-01,entregado,telefono,6,3222,2750,19332,16500,2832
3,1,79,6,2024-01-01,entregado,telefono,24,3555,2850,85320,68400,16920
4,2,112,68,2024-01-01,entregado,web,6,1370,900,8220,5400,2820
5,2,38,68,2024-01-01,entregado,web,6,1490,1150,8940,6900,2040
6,2,130,68,2024-01-01,entregado,web,12,3515,2850,42180,34200,7980
7,3,19,17,2024-01-01,entregado,terreno,48,2380,2100,114240,100800,13440
8,3,13,17,2024-01-01,entregado,terreno,36,5480,3650,197280,131400,65880
9,3,119,17,2024-01-01,entregado,terreno,18,2888,2000,51984,36000,15984
10,4,2,67,2024-01-01,entregado,terreno,24,2770,1850,66480,44400,22080


In [51]:
%%sql

WITH base_analitica AS (
   SELECT
    dp.detalle_id,
    p.pedido_id,
    dp.producto_id,
    p.cliente_id,
    p.fecha_pedido,
    p.estado,
    p.canal,
    dp.cantidad,
    dp.neto_unitario_clp,
    dp.costo_unitario_clp,
    dp.neto_unitario_clp * dp.cantidad AS importe_neto_linea_clp,
    dp.costo_unitario_clp * dp.cantidad AS costo_linea_clp,
    dp.neto_unitario_clp * dp.cantidad - dp.costo_unitario_clp * dp.cantidad AS margen_bruto_estimado_linea_clp
FROM detalle_pedidos AS dp
INNER JOIN pedidos AS p ON dp.pedido_id = p.pedido_id
)
SELECT
    COUNT(*) AS total_filas,
    COUNT(DISTINCT detalle_id) AS detalle_distintos,
    SUM(importe_neto_linea_clp) AS importe_neto_total_clp
FROM base_analitica;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_filas,detalle_distintos,importe_neto_total_clp
61647,61647,4666686966



## 2.2 Construcción de la tabla de hechos comercial

### 2.2.1 Fuente analítica preliminar

**Objetivo**

Construir una fuente analítica a nivel de línea de pedido que integre los atributos comerciales necesarios para el cálculo de cantidades, importes netos, costos y márgenes brutos estimados.

**Granularidad**

Cada fila representa una línea de pedido identificada por `detalle_id`.

**Metodología**

La fuente se construye a partir de `detalle_pedidos`, incorporando información de `pedidos` mediante `pedido_id`.

Se calculan tres medidas monetarias a nivel de línea:

- `importe_neto_linea_clp`: cantidad multiplicada por neto unitario.
- `costo_linea_clp`: cantidad multiplicada por costo unitario.
- `margen_bruto_estimado_linea_clp`: importe neto de línea menos costo de línea.

Se conservan los pedidos entregados, pendientes y cancelados para permitir análisis diferenciados por estado comercial.



### 2.2.2 Validación de granularidad y conciliación

Se verificó la consistencia de la fuente analítica preliminar mediante controles de volumen, unicidad e importes monetarios.

| Indicador | Resultado | Referencia esperada |
|---|---:|---:|
| Total de filas | 61.647 | 61.647 |
| Identificadores distintos | 61.647 | 61.647 |
| Importe neto total (CLP) | $4.666.686.966 | $4.666.686.966 |

#### Hallazgos

La consulta conserva la totalidad de las líneas de detalle originales sin detectar duplicaciones en sus identificadores.

El importe neto agregado coincide con el total obtenido en la auditoría inicial.

No se identificaron pérdidas de registros ni diferencias monetarias derivadas de la relación entre pedidos y líneas de detalle.

#### Conclusión

La fuente analítica preliminar supera los controles de granularidad, unicidad y conciliación monetaria establecidos.

Estos resultados respaldan su utilización como base para una tabla de hechos comercial.

Los importes incluyen pedidos entregados, pendientes y cancelados, por lo que deben interpretarse según su estado comercial antes de utilizarlos en indicadores de desempeño.

**Estado del control:** conforme.



### 2.2.3 Implementación de la vista analítica

#### Objetivo

Implementar una vista SQL reutilizable que represente la tabla de hechos comercial a nivel de línea de pedido.

#### Diseño

La vista `fact_detalle_pedidos` conservará una fila por `detalle_id` e incorporará atributos de pedido y medidas monetarias calculadas.

Se mantendrán los tres estados comerciales: entregado, pendiente y cancelado.

#### Implementación

La vista se creará sobre una copia de trabajo de la base SQLite, manteniendo intacta la fuente operacional original.

La definición SQL se conservará además en un archivo independiente para facilitar su mantenimiento y reproducibilidad.

#### Consideraciones

Las vistas de SQLite no almacenan físicamente sus resultados.

Los datos se consultarán desde las tablas de origen al ejecutar la vista.

Las medidas monetarias deberán interpretarse de acuerdo con el estado comercial de cada pedido.


In [52]:
from pathlib import Path

db_analytics = Path("../data/cordillera_analytics.sqlite").resolve()

print("Base analítica encontrada:", db_analytics.is_file())

Base analítica encontrada: True


In [53]:
%sql sqlite:///../data/cordillera_analytics.sqlite

Switching to connection 'sqlite:///../data/cordillera_analytics.sqlite'

In [ ]:
%%sql

CREATE VIEW IF NOT EXISTS fact_detalle_pedidos AS
SELECT
    dp.detalle_id,
    p.pedido_id,
    dp.producto_id,
    p.cliente_id,
    p.fecha_pedido,
    p.estado,
    p.canal,
    dp.cantidad,
    dp.neto_unitario_clp,
    dp.costo_unitario_clp,
    dp.neto_unitario_clp * dp.cantidad AS importe_neto_linea_clp,
    dp.costo_unitario_clp * dp.cantidad AS costo_linea_clp,
    dp.neto_unitario_clp * dp.cantidad - dp.costo_unitario_clp * dp.cantidad AS margen_bruto_estimado_linea_clp
FROM detalle_pedidos AS dp
INNER JOIN pedidos AS p ON dp.pedido_id = p.pedido_id;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

++
||
++
++

In [55]:

%%sql

SELECT
    name,
    type
FROM sqlite_master
WHERE type = 'view'
    AND name = 'fact_detalle_pedidos';

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

name,type
fact_detalle_pedidos,view


In [56]:
%%sql

SELECT *
FROM fact_detalle_pedidos
ORDER BY detalle_id ASC
LIMIT 5;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

detalle_id,pedido_id,producto_id,cliente_id,fecha_pedido,estado,canal,cantidad,neto_unitario_clp,costo_unitario_clp,importe_neto_linea_clp,costo_linea_clp,margen_bruto_estimado_linea_clp
1,1,168,6,2024-01-01,entregado,telefono,36,4513,3400,162468,122400,40068
2,1,110,6,2024-01-01,entregado,telefono,6,3222,2750,19332,16500,2832
3,1,79,6,2024-01-01,entregado,telefono,24,3555,2850,85320,68400,16920
4,2,112,68,2024-01-01,entregado,web,6,1370,900,8220,5400,2820
5,2,38,68,2024-01-01,entregado,web,6,1490,1150,8940,6900,2040


In [57]:
%%sql

SELECT
    COUNT(*) AS total_filas,
    COUNT(DISTINCT detalle_id) AS detalle_distintos,
    SUM(importe_neto_linea_clp) AS importe_neto_total_clp,
    SUM(CASE WHEN estado = 'entregado' THEN importe_neto_linea_clp ELSE 0 END) AS importe_neto_entregado_clp
FROM fact_detalle_pedidos;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_filas,detalle_distintos,importe_neto_total_clp,importe_neto_entregado_clp
61647,61647,4666686966,4434202034



### 2.2.4 Resultados de implementación y validación

Se implementó la vista `fact_detalle_pedidos` en la base de trabajo `cordillera_analytics.sqlite`, a partir de las tablas operacionales `detalle_pedidos` y `pedidos`.

La vista representa una línea de pedido por registro y contiene atributos comerciales y medidas monetarias calculadas.

#### Resultados de validación

| Indicador | Resultado |
|---|---:|
| Filas totales | 61.647 |
| Identificadores de detalle distintos | 61.647 |
| Importe neto total (CLP) | $4.666.686.966 |
| Importe neto de pedidos entregados (CLP) | $4.434.202.034 |

#### Hallazgos

- La vista fue creada y consultada correctamente.
- Se conservaron las 61.647 líneas de detalle sin duplicaciones de identificadores.
- Los importes calculados coinciden con las referencias obtenidas durante la auditoría inicial.
- La vista permite reutilizar los atributos y las medidas comerciales sin reconstruir las relaciones operacionales en cada consulta.

#### Consideraciones técnicas

La vista conserva los tres estados comerciales, permitiendo diferenciar los importes de pedidos entregados, pendientes y cancelados.

Las medidas monetarias se calculan a nivel de línea de pedido. El margen bruto estimado no representa utilidad neta ni incorpora gastos indirectos.

#### Conclusión

La vista `fact_detalle_pedidos` supera los controles de volumen, unicidad y conciliación monetaria establecidos.

Su implementación constituye la primera fuente reutilizable del modelo analítico comercial.

**Estado del control:** conforme.



## 2.3 Diseño e implementación de dimensiones

### 2.3.1 Dimensión Cliente

#### Objetivo

Diseñar una dimensión que permita contextualizar los hechos comerciales mediante atributos descriptivos, geográficos y de segmentación de clientes.

#### Granularidad

Cada fila representará un cliente identificado de forma única mediante `cliente_id`.

#### Metodología

Se seleccionarán los atributos relevantes de la tabla operacional `clientes`, conservando una estructura adecuada para establecer una relación de uno a muchos con `fact_detalle_pedidos`.

Se verificará que la dimensión mantenga la unicidad de sus identificadores y que sus claves sean consistentes con las referencias utilizadas por la tabla de hechos.

La implementación definitiva se realizará mediante una vista SQL reutilizable.


In [58]:
%%sql

SELECT 
    cliente_id,
    razon_social,
    segmento,
    region,
    comuna,
    fecha_alta,
    vendedor_id,
    plazo_dias
FROM clientes;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

cliente_id,razon_social,segmento,region,comuna,fecha_alta,vendedor_id,plazo_dias
1,Almacen Ficticio Cordillera 0001,Almacen,Metropolitana,Puente Alto,2024-01-01,4,0
2,Horeca Ficticio Cordillera 0002,Horeca,Metropolitana,Maipu,2024-01-01,11,15
3,Almacen Ficticio Cordillera 0003,Almacen,Metropolitana,Puente Alto,2024-01-01,1,30
4,Almacen Ficticio Cordillera 0004,Almacen,Metropolitana,Puente Alto,2024-01-01,6,30
5,Almacen Ficticio Cordillera 0005,Almacen,Metropolitana,Santiago,2024-01-01,3,15
6,Almacen Ficticio Cordillera 0006,Almacen,Valparaiso,Vina del Mar,2024-01-01,11,30
7,Almacen Ficticio Cordillera 0007,Almacen,Metropolitana,Maipu,2024-01-01,12,0
8,Almacen Ficticio Cordillera 0008,Almacen,Metropolitana,Santiago,2024-01-01,2,0
9,Almacen Ficticio Cordillera 0009,Almacen,Metropolitana,Santiago,2024-01-01,9,30
10,Minimarket Ficticio Cordillera 0010,Minimarket,Metropolitana,Puente Alto,2024-01-01,2,0


In [59]:
%%sql

WITH base_clientes AS (
    SELECT 
    cliente_id,
    razon_social,
    segmento,
    region,
    comuna,
    fecha_alta,
    vendedor_id,
    plazo_dias
FROM clientes
)
SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT cliente_id) AS clientes_distintos,
    SUM(CASE WHEN cliente_id IS NULL THEN 1 ELSE 0 END) AS clientes_id_nulos
FROM base_clientes

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_clientes,clientes_distintos,clientes_id_nulos
600,600,0



#### 2.3.1.1 Validación de la dimensión Cliente

Se diseñó una fuente analítica de clientes a partir de la tabla operacional `clientes`, seleccionando atributos comerciales, geográficos y de condiciones de crédito.

La dimensión propuesta conserva una granularidad de una fila por `cliente_id`, independientemente de la actividad comercial registrada.

##### Resultados

| Indicador | Resultado |
|---|---:|
| Total de registros | 600 |
| Clientes distintos | 600 |
| Identificadores nulos | 0 |

##### Hallazgos

No se detectaron identificadores duplicados ni valores nulos en la clave de relación.

La fuente conserva la totalidad de los clientes registrados, incluidos aquellos que no presentan pedidos asociados.

##### Decisiones de diseño

- Se utilizará `cliente_id` como clave de relación con la tabla de hechos.
- Los atributos `segmento`, `region` y `comuna` permitirán realizar análisis comerciales y geográficos.
- `fecha_alta` y `plazo_dias` permitirán analizar antigüedad y condiciones de crédito.
- Se conservará `vendedor_id` como referencia comercial para futuras relaciones.
- Se excluirá `email` por no ser necesario para los indicadores previstos.

La dimensión representará los atributos de clientes disponibles en el estado actual de la fuente, sin implementar todavía mecanismos de historización de cambios.

##### Conclusión

La fuente propuesta supera los controles iniciales de granularidad y unicidad, por lo que se considera apta para su implementación como dimensión analítica.

**Estado del control:** conforme.


In [60]:
%%sql

CREATE VIEW IF NOT EXISTS dim_cliente AS
SELECT 
    cliente_id,
    razon_social,
    segmento,
    region,
    comuna,
    fecha_alta,
    vendedor_id,
    plazo_dias
FROM clientes;



Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

++
||
++
++

In [61]:

%%sql

SELECT
    name,
    type
FROM sqlite_master
WHERE type = 'view'
    AND name = 'dim_cliente';

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

name,type
dim_cliente,view


In [62]:
%%sql

SELECT *
FROM dim_cliente
LIMIT 5;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

cliente_id,razon_social,segmento,region,comuna,fecha_alta,vendedor_id,plazo_dias
1,Almacen Ficticio Cordillera 0001,Almacen,Metropolitana,Puente Alto,2024-01-01,4,0
2,Horeca Ficticio Cordillera 0002,Horeca,Metropolitana,Maipu,2024-01-01,11,15
3,Almacen Ficticio Cordillera 0003,Almacen,Metropolitana,Puente Alto,2024-01-01,1,30
4,Almacen Ficticio Cordillera 0004,Almacen,Metropolitana,Puente Alto,2024-01-01,6,30
5,Almacen Ficticio Cordillera 0005,Almacen,Metropolitana,Santiago,2024-01-01,3,15


In [63]:
%%sql

SELECT 
    COUNT(*) AS total_registros,
    COUNT(DISTINCT cliente_id) AS clientes_distintos
FROM dim_cliente;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_registros,clientes_distintos
600,600



#### 2.3.1.2 Implementación y validación de `dim_cliente`

Se implementó la vista `dim_cliente` en la base analítica `cordillera_analytics.sqlite`, utilizando atributos seleccionados de la tabla operacional `clientes`.

La vista conserva una fila por cliente e incluye información comercial, geográfica y de condiciones de crédito.

##### Resultados de validación

| Indicador | Resultado |
|---|---:|
| Total de registros | 600 |
| Identificadores distintos | 600 |
| Identificadores nulos | 0 |
| Vista disponible en SQLite | Sí |

##### Conclusión

La dimensión Cliente fue implementada correctamente y conserva la granularidad definida, sin identificar duplicaciones ni claves ausentes.

Se establece `cliente_id` como clave de relación lógica con `fact_detalle_pedidos`, mediante una cardinalidad de uno a muchos (1:N).

La dimensión incluye la totalidad de la cartera registrada, independientemente de su actividad comercial.

**Estado del componente:** implementado y validado.



### 2.3.2 Dimensión Producto

#### Objetivo

Diseñar una dimensión analítica que integre los atributos descriptivos y comerciales de los productos con su clasificación por categoría.

#### Granularidad

Cada registro representará un producto identificado de manera única mediante `producto_id`.

#### Metodología

Se examinarán las estructuras de `productos` y `categorias` para identificar sus claves, atributos y relación.

Posteriormente, se construirá una fuente integrada que conserve la totalidad de los productos registrados, evitando duplicaciones y pérdidas de información.

La dimensión permitirá analizar los indicadores comerciales por producto y categoría.

#### Consideraciones

Los atributos comerciales de productos representan los valores disponibles en la fuente al momento de consulta y no necesariamente sus valores históricos al momento de cada pedido.

Los importes y costos de las operaciones se calcularán desde la tabla de hechos, utilizando los valores registrados en cada línea.


In [64]:
%%sql

PRAGMA table_info(categorias);

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

cid,name,type,notnull,dflt_value,pk
0,categoria_id,INTEGER,0,None,1
1,nombre,TEXT,1,None,0


In [65]:
%%sql

SELECT *
FROM categorias
LIMIT 10;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

categoria_id,nombre
1,Abarrotes
2,Bebidas
3,Limpieza
4,Desechables
5,Snacks
6,Cafeteria


In [66]:
%%sql

SELECT
    p.producto_id,
    p.sku,
    p.nombre AS nombre_producto,
    p.categoria_id,
    c.nombre AS nombre_categoria,
    p.proveedor_id,
    p.unidad_venta,
    p.precio_lista_clp,
    p.costo_referencia_clp,
    p.stock_minimo,
    p.fecha_alta
FROM productos AS p
LEFT JOIN categorias AS c
ON p.categoria_id = c.categoria_id;


Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

producto_id,sku,nombre_producto,categoria_id,nombre_categoria,proveedor_id,unidad_venta,precio_lista_clp,costo_referencia_clp,stock_minimo,fecha_alta
1,COR-0001,Arroz 1 kg Marca Ficticia 1,1,Abarrotes,2,unidad comercial,4400,3350,75,2024-01-01
2,COR-0002,Fideos 400 g Marca Ficticia 1,1,Abarrotes,3,unidad comercial,2770,1850,81,2024-01-01
3,COR-0003,Aceite 1 L Marca Ficticia 1,1,Abarrotes,1,unidad comercial,5730,3600,81,2024-01-01
4,COR-0004,Azucar 1 kg Marca Ficticia 1,1,Abarrotes,2,unidad comercial,6280,4000,52,2024-01-01
5,COR-0005,Harina 1 kg Marca Ficticia 1,1,Abarrotes,3,unidad comercial,4400,3100,55,2024-01-01
6,COR-0006,Arroz 1 kg Marca Ficticia 2,1,Abarrotes,1,unidad comercial,4130,2600,40,2024-01-01
7,COR-0007,Fideos 400 g Marca Ficticia 2,1,Abarrotes,2,unidad comercial,1680,1050,45,2024-01-01
8,COR-0008,Aceite 1 L Marca Ficticia 2,1,Abarrotes,3,unidad comercial,2930,1850,45,2024-01-01
9,COR-0009,Azucar 1 kg Marca Ficticia 2,1,Abarrotes,1,unidad comercial,2650,2050,69,2024-01-01
10,COR-0010,Harina 1 kg Marca Ficticia 2,1,Abarrotes,2,unidad comercial,5020,3800,73,2024-01-01


In [67]:
%%sql

WITH base_productos AS (
    SELECT 
    p.producto_id,
    p.sku,
    p.nombre AS nombre_producto,
    p.categoria_id,
    c.nombre AS nombre_categoria,
    p.proveedor_id,
    p.unidad_venta,
    p.precio_lista_clp,
    p.costo_referencia_clp,
    p.stock_minimo,
    p.fecha_alta
FROM productos AS p
LEFT JOIN categorias AS c
ON p.categoria_id = c.categoria_id
)
SELECT 
    COUNT(*) AS total_productos,
    COUNT(DISTINCT producto_id) AS productos_distintos,
    SUM(CASE WHEN nombre_categoria IS NULL THEN 1 ELSE 0 END) AS productos_sin_categoria
FROM base_productos;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_productos,productos_distintos,productos_sin_categoria
180,180,0



#### 2.3.2.1 Validación de la dimensión Producto

Se construyó una fuente analítica que integra los atributos del catálogo `productos` con la clasificación disponible en `categorias`.

La relación se estableció mediante `categoria_id`, utilizando `LEFT JOIN` para conservar todos los productos registrados.

##### Resultados de validación

| Indicador | Resultado |
|---|---:|
| Total de productos | 180 |
| Identificadores distintos | 180 |
| Productos sin categoría asociada | 0 |

##### Hallazgos

La integración conserva la totalidad de los 180 productos registrados, sin generar duplicaciones de identificadores.

Todos los productos presentan una categoría asociada con nombre informado.

La fuente mantiene una granularidad de una fila por `producto_id`.

##### Decisiones de diseño

- Se utilizará `producto_id` como clave de relación con `fact_detalle_pedidos`.
- Los atributos `nombre_producto` y `nombre_categoria` permitirán analizar el desempeño comercial por producto y categoría.
- Se conservarán los atributos de proveedor, unidad de venta, precio de lista, costo de referencia y stock mínimo.
- Los indicadores monetarios históricos se calcularán utilizando los valores registrados en la tabla de hechos, no los valores de referencia del catálogo.

##### Conclusión

La fuente integrada supera los controles iniciales de unicidad, completitud de categorías y conservación de registros.

Se considera apta para su implementación como dimensión analítica.

**Estado del control:** conforme.


In [68]:
%%sql

CREATE VIEW IF NOT EXISTS dim_producto AS
SELECT
        p.producto_id,
        p.sku,
        p.nombre AS nombre_producto,
        p.categoria_id,
        c.nombre AS nombre_categoria,
        p.proveedor_id,
        p.unidad_venta,
        p.precio_lista_clp,
        p.costo_referencia_clp,
        p.stock_minimo,
        p.fecha_alta
FROM productos AS p
LEFT JOIN categorias AS c
    ON p.categoria_id = c.categoria_id;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

++
||
++
++

In [69]:

%%sql

SELECT
    name,
    type
FROM sqlite_master
WHERE type = 'view'
    AND name = 'dim_producto';

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

name,type
dim_producto,view


In [70]:
%%sql

SELECT *
FROM dim_producto
LIMIT 5;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

producto_id,sku,nombre_producto,categoria_id,nombre_categoria,proveedor_id,unidad_venta,precio_lista_clp,costo_referencia_clp,stock_minimo,fecha_alta
1,COR-0001,Arroz 1 kg Marca Ficticia 1,1,Abarrotes,2,unidad comercial,4400,3350,75,2024-01-01
2,COR-0002,Fideos 400 g Marca Ficticia 1,1,Abarrotes,3,unidad comercial,2770,1850,81,2024-01-01
3,COR-0003,Aceite 1 L Marca Ficticia 1,1,Abarrotes,1,unidad comercial,5730,3600,81,2024-01-01
4,COR-0004,Azucar 1 kg Marca Ficticia 1,1,Abarrotes,2,unidad comercial,6280,4000,52,2024-01-01
5,COR-0005,Harina 1 kg Marca Ficticia 1,1,Abarrotes,3,unidad comercial,4400,3100,55,2024-01-01


In [71]:
%%sql

SELECT 
    COUNT(*) AS total_registros,
    COUNT(DISTINCT producto_id) AS productos_distintos,
    SUM(CASE WHEN nombre_categoria IS NULL THEN 1 ELSE 0 END) AS productos_sin_categoria
FROM dim_producto;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_registros,productos_distintos,productos_sin_categoria
180,180,0



#### 2.3.2.2 Implementación y validación de `dim_producto`

Se implementó la vista `dim_producto` en `cordillera_analytics.sqlite`, integrando las tablas operacionales `productos` y `categorias` mediante `categoria_id`.

La dimensión conserva una fila por producto e incorpora atributos descriptivos, de clasificación y de referencia comercial.

##### Resultados de validación

| Indicador | Resultado |
|---|---:|
| Total de registros | 180 |
| Identificadores distintos | 180 |
| Productos sin categoría asociada | 0 |
| Vista disponible en SQLite | Sí |

##### Hallazgos

La integración conserva los 180 productos registrados sin generar duplicaciones.

Todos los productos cuentan con una categoría asociada con nombre informado.

##### Decisiones de diseño

- Se establece `producto_id` como clave de relación lógica con `fact_detalle_pedidos`.
- Se incorpora el nombre de categoría para facilitar los análisis comerciales por agrupación de productos.
- Se conservan atributos de referencia, como precio de lista, costo de referencia y stock mínimo.
- Las medidas monetarias históricas se calcularán utilizando los valores almacenados en las líneas de pedido.

##### Conclusión

La vista `dim_producto` supera los controles de unicidad, conservación de registros y completitud de categorías.

Se considera apta para integrar el modelo dimensional comercial.

**Estado del componente:** implementado y validado.



### 2.3.3 Dimensión Fecha

#### Objetivo

Diseñar una dimensión temporal reutilizable que permita analizar los hechos comerciales mediante atributos de calendario y períodos de tiempo.

#### Granularidad

Cada registro representará una fecha calendario única.

#### Metodología

Se examinará la tabla operacional `calendario` para determinar su estructura, cobertura temporal y disponibilidad de atributos.

Se evaluará su utilización como fuente para `dim_fecha`, considerando la necesidad de mantener una fecha única por registro y una cobertura compatible con las fechas de los pedidos.

#### Consideraciones

Inicialmente, la dimensión se relacionará con `fact_detalle_pedidos` mediante `fecha_pedido`.

Las fechas de entrega y vencimiento corresponden a roles temporales distintos, cuya incorporación al modelo se evaluará posteriormente.

La dimensión debe permitir comparaciones y agrupaciones cronológicas consistentes, sin modificar la granularidad de los hechos comerciales.


In [72]:
%%sql

PRAGMA table_info(calendario);

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

cid,name,type,notnull,dflt_value,pk
0,fecha,TEXT,0,None,1
1,anio,INTEGER,1,None,0
2,mes,INTEGER,1,None,0
3,dia,INTEGER,1,None,0
4,dia_semana,INTEGER,1,None,0
5,mes_inicio,TEXT,1,None,0


In [73]:
%%sql

SELECT *
FROM calendario
LIMIT 10;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

fecha,anio,mes,dia,dia_semana,mes_inicio
2024-01-01,2024,1,1,1,2024-01-01
2024-01-02,2024,1,2,2,2024-01-01
2024-01-03,2024,1,3,3,2024-01-01
2024-01-04,2024,1,4,4,2024-01-01
2024-01-05,2024,1,5,5,2024-01-01
2024-01-06,2024,1,6,6,2024-01-01
2024-01-07,2024,1,7,7,2024-01-01
2024-01-08,2024,1,8,1,2024-01-01
2024-01-09,2024,1,9,2,2024-01-01
2024-01-10,2024,1,10,3,2024-01-01


In [74]:
%%sql

SELECT
    COUNT(*) AS total_fechas,
    COUNT(DISTINCT fecha) AS fechas_distintas,
    MIN(fecha) AS fecha_minima,
    MAX(fecha) AS fecha_maxima,
    SUM(CASE WHEN fecha IS NULL THEN 1 ELSE 0 END) AS fechas_nulas
FROM calendario;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_fechas,fechas_distintas,fecha_minima,fecha_maxima,fechas_nulas
1004,1004,2024-01-01,2026-09-30,0


In [75]:
%%sql

SELECT
    COUNT(*) AS lineas_sin_calendario,
    COUNT(DISTINCT f.fecha_pedido) AS fechas_sin_calendario
FROM fact_detalle_pedidos AS f
LEFT JOIN calendario AS c
ON c.fecha = f.fecha_pedido
WHERE c.fecha IS NULL;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

lineas_sin_calendario,fechas_sin_calendario
0,0



#### 2.3.3.1 Validación de integridad y cobertura temporal

Se evaluó la estructura y cobertura de la tabla `calendario`, considerando su posible utilización como dimensión temporal del modelo analítico.

La validación incluyó controles de unicidad, completitud, extensión del calendario y correspondencia con las fechas de creación de los pedidos registrados en `fact_detalle_pedidos`.

##### Resultados

| Indicador | Resultado |
|---|---:|
| Total de fechas | 1.004 |
| Fechas distintas | 1.004 |
| Fechas nulas | 0 |
| Fecha mínima | 2024-01-01 |
| Fecha máxima | 2026-09-30 |
| Líneas sin correspondencia temporal | 0 |
| Fechas distintas sin correspondencia | 0 |

##### Hallazgos

El calendario presenta 1.004 fechas únicas y consecutivas, correspondientes al período comprendido entre enero de 2024 y septiembre de 2026.

No se identificaron fechas nulas ni duplicadas.

La totalidad de las líneas registradas en `fact_detalle_pedidos` encuentra correspondencia entre `fecha_pedido` y `calendario.fecha`.

##### Decisiones de diseño

Se utilizará `calendario` como fuente para construir `dim_fecha`.

La dimensión conservará una fila por fecha e incorporará atributos complementarios para facilitar el análisis temporal.

Inicialmente, la relación con la tabla de hechos se establecerá mediante la fecha de creación del pedido.

##### Conclusión

La fuente temporal supera los controles de integridad y cobertura aplicados, por lo que se considera apta para implementar la dimensión Fecha.

**Estado del control:** conforme.


In [76]:
%%sql

SELECT
     fecha,
     anio,
     mes,
     dia,
     dia_semana,
     mes_inicio,
     STRFTIME('%Y-%m', fecha) AS anio_mes,
     CASE 
        WHEN mes IN (1,2,3) THEN 1
        WHEN mes IN (4,5,6) THEN 2
        WHEN mes IN (7,8,9) THEN 3
        WHEN mes IN (10,11,12) THEN 4
        ELSE NULL END AS trimestre,
    CASE 
        WHEN mes = 1 THEN 'Enero'
        WHEN mes = 2 THEN 'Febrero'
        WHEN mes = 3 THEN 'Marzo'
        WHEN mes = 4 THEN 'Abril'
        WHEN mes = 5 THEN 'Mayo'
        WHEN mes = 6 THEN 'Junio'
        WHEN mes = 7 THEN 'Julio'
        WHEN mes = 8 THEN 'Agosto'
        WHEN mes = 9 THEN 'Septiembre'
        WHEN mes = 10 THEN 'Octubre'
        WHEN mes = 11 THEN 'Noviembre'
        WHEN mes = 12 THEN 'Diciembre'
        ELSE NULL END AS nombre_mes,
    CASE
        WHEN dia_semana = 1 THEN 'Lunes'
        WHEN dia_semana = 2 THEN 'Martes'
        WHEN dia_semana = 3 THEN 'Miércoles'
        WHEN dia_semana = 4 THEN 'Jueves'
        WHEN dia_semana = 5 THEN 'Viernes'
        WHEN dia_semana = 6 THEN 'Sábado'
        WHEN dia_semana = 7 THEN 'Domingo' 
        ELSE NULL END AS nombre_dia_semana
FROM calendario
WHERE fecha IN (
    '2024-01-01',
    '2024-04-01',
    '2024-07-01',
    '2024-10-01',
    '2025-03-14',
    '2026-09-30'
)
ORDER BY fecha ASC;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

fecha,anio,mes,dia,dia_semana,mes_inicio,anio_mes,trimestre,nombre_mes,nombre_dia_semana
2024-01-01,2024,1,1,1,2024-01-01,2024-01,1,Enero,Lunes
2024-04-01,2024,4,1,1,2024-04-01,2024-04,2,Abril,Lunes
2024-07-01,2024,7,1,1,2024-07-01,2024-07,3,Julio,Lunes
2024-10-01,2024,10,1,2,2024-10-01,2024-10,4,Octubre,Martes
2025-03-14,2025,3,14,5,2025-03-01,2025-03,1,Marzo,Viernes
2026-09-30,2026,9,30,3,2026-09-01,2026-09,3,Septiembre,Miércoles


In [79]:
%%sql

WITH base_fechas AS (
    SELECT
     fecha,
     anio,
     mes,
     dia,
     dia_semana,
     mes_inicio,
     STRFTIME('%Y-%m', fecha) AS anio_mes,
     CASE 
        WHEN mes IN (1,2,3) THEN 1
        WHEN mes IN (4,5,6) THEN 2
        WHEN mes IN (7,8,9) THEN 3
        WHEN mes IN (10,11,12) THEN 4
        ELSE NULL END AS trimestre,
    CASE 
        WHEN mes = 1 THEN 'Enero'
        WHEN mes = 2 THEN 'Febrero'
        WHEN mes = 3 THEN 'Marzo'
        WHEN mes = 4 THEN 'Abril'
        WHEN mes = 5 THEN 'Mayo'
        WHEN mes = 6 THEN 'Junio'
        WHEN mes = 7 THEN 'Julio'
        WHEN mes = 8 THEN 'Agosto'
        WHEN mes = 9 THEN 'Septiembre'
        WHEN mes = 10 THEN 'Octubre'
        WHEN mes = 11 THEN 'Noviembre'
        WHEN mes = 12 THEN 'Diciembre'
        ELSE NULL END AS nombre_mes,
    CASE
        WHEN dia_semana = 1 THEN 'Lunes'
        WHEN dia_semana = 2 THEN 'Martes'
        WHEN dia_semana = 3 THEN 'Miércoles'
        WHEN dia_semana = 4 THEN 'Jueves'
        WHEN dia_semana = 5 THEN 'Viernes'
        WHEN dia_semana = 6 THEN 'Sábado'
        WHEN dia_semana = 7 THEN 'Domingo' 
        ELSE NULL END AS nombre_dia_semana
FROM calendario
)   
SELECT 
    COUNT(*) AS total_fechas,
    COUNT(DISTINCT fecha) AS fechas_distintas,
    SUM(CASE WHEN nombre_mes IS NULL THEN 1 ELSE 0 END) AS meses_sin_nombre,
    SUM(CASE WHEN nombre_dia_semana IS NULL THEN 1 ELSE 0 END) AS dias_sin_nombre,
    SUM(CASE WHEN trimestre IS NULL THEN 1 ELSE 0 END) AS trimestres_invalidos
FROM base_fechas;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_fechas,fechas_distintas,meses_sin_nombre,dias_sin_nombre,trimestres_invalidos
1004,1004,0,0,0


In [81]:
%%sql

CREATE VIEW IF NOT EXISTS dim_fecha AS
SELECT
     fecha,
     anio,
     mes,
     dia,
     dia_semana,
     mes_inicio,
     STRFTIME('%Y-%m', fecha) AS anio_mes,
     CASE 
        WHEN mes IN (1,2,3) THEN 1
        WHEN mes IN (4,5,6) THEN 2
        WHEN mes IN (7,8,9) THEN 3
        WHEN mes IN (10,11,12) THEN 4
        ELSE NULL END AS trimestre,
    CASE 
        WHEN mes = 1 THEN 'Enero'
        WHEN mes = 2 THEN 'Febrero'
        WHEN mes = 3 THEN 'Marzo'
        WHEN mes = 4 THEN 'Abril'
        WHEN mes = 5 THEN 'Mayo'
        WHEN mes = 6 THEN 'Junio'
        WHEN mes = 7 THEN 'Julio'
        WHEN mes = 8 THEN 'Agosto'
        WHEN mes = 9 THEN 'Septiembre'
        WHEN mes = 10 THEN 'Octubre'
        WHEN mes = 11 THEN 'Noviembre'
        WHEN mes = 12 THEN 'Diciembre'
        ELSE NULL END AS nombre_mes,
    CASE
        WHEN dia_semana = 1 THEN 'Lunes'
        WHEN dia_semana = 2 THEN 'Martes'
        WHEN dia_semana = 3 THEN 'Miércoles'
        WHEN dia_semana = 4 THEN 'Jueves'
        WHEN dia_semana = 5 THEN 'Viernes'
        WHEN dia_semana = 6 THEN 'Sábado'
        WHEN dia_semana = 7 THEN 'Domingo' 
        ELSE NULL END AS nombre_dia_semana
FROM calendario;


Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

++
||
++
++

In [83]:
%%sql

SELECT 
    COUNT(*) AS total_fechas,
    COUNT(DISTINCT fecha) AS fechas_distintas,
    SUM(CASE WHEN nombre_mes IS NULL THEN 1 ELSE 0 END) AS meses_sin_nombre,
    SUM(CASE WHEN nombre_dia_semana IS NULL THEN 1 ELSE 0 END) AS dias_sin_nombre,
    SUM(CASE WHEN trimestre IS NULL THEN 1 ELSE 0 END) AS trimestres_invalidos
FROM dim_fecha;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_fechas,fechas_distintas,meses_sin_nombre,dias_sin_nombre,trimestres_invalidos
1004,1004,0,0,0



#### 2.3.3.2 Implementación y validación de `dim_fecha`

Se implementó la vista `dim_fecha` en la base analítica `cordillera_analytics.sqlite`, utilizando como fuente la tabla `calendario`.

La dimensión conserva una granularidad de una fila por fecha e incorpora atributos complementarios para facilitar el análisis temporal.

##### Atributos incorporados

Además de los campos originales, se incorporaron:

- `anio_mes`: período en formato YYYY-MM.
- `trimestre`: trimestre calendario, del 1 al 4.
- `nombre_mes`: denominación del mes en español.
- `nombre_dia_semana`: denominación del día en español.

##### Resultados de validación

| Indicador | Resultado |
|---|---:|
| Total de fechas | 1.004 |
| Fechas distintas | 1.004 |
| Fechas sin nombre de mes | 0 |
| Fechas sin nombre de día | 0 |
| Trimestres inválidos | 0 |
| Fechas de pedido sin correspondencia | 0 |

##### Hallazgos

La dimensión conserva las 1.004 fechas del calendario original, comprendidas entre el 1 de enero de 2024 y el 30 de septiembre de 2026.

Los atributos calculados presentan resultados completos dentro de los controles realizados.

La totalidad de las fechas de creación de pedidos contenidas en `fact_detalle_pedidos` encuentra correspondencia en el calendario.

##### Decisiones de diseño

- Se utilizará `fecha` como clave de relación lógica con `fact_detalle_pedidos.fecha_pedido`.
- Los atributos numéricos de mes y día de semana permitirán ordenar correctamente sus denominaciones en Power BI.
- Inicialmente, la dimensión representará la fecha de creación de los pedidos.
- Los otros roles temporales, como entrega y vencimiento, requerirán definiciones de relación y medidas específicas.

##### Conclusión

La dimensión Fecha fue implementada y validada, conservando la granularidad establecida y una cobertura temporal compatible con los hechos comerciales examinados.

**Estado del componente:** implementado y validado.



## 2.4 Validación integral del modelo dimensional

### 2.4.1 Integridad de relaciones y conservación de granularidad

#### Objetivo

Verificar la integridad de las relaciones entre la tabla de hechos comercial y las dimensiones Cliente, Producto y Fecha.

#### Metodología

Se relacionará `fact_detalle_pedidos` con las tres dimensiones mediante sus claves correspondientes.

Se utilizarán relaciones que permitan conservar todas las líneas de hechos, identificando posibles referencias sin correspondencia.

La validación evaluará:

1. Conservación de registros y unicidad de identificadores.
2. Correspondencia de claves con las dimensiones.
3. Consistencia de los importes monetarios después de incorporar las dimensiones.

Los resultados se contrastarán con las referencias establecidas durante la auditoría y construcción de la tabla de hechos.


In [87]:
%%sql

SELECT
    COUNT(*) total_filas,
    COUNT(DISTINCT t.detalle_id) AS detalles_distintos,
    SUM(CASE WHEN c.cliente_id IS NULL THEN 1 ELSE 0 END) AS lineas_sin_cliente,
    SUM(CASE WHEN p.producto_id IS NULL THEN 1 ELSE 0 END) AS lineas_sin_producto,
    SUM(CASE WHEN f.fecha IS NULL THEN 1 ELSE 0 END) AS lineas_sin_fecha,
    SUM(t.importe_neto_linea_clp) AS importe_neto_total_clp
FROM fact_detalle_pedidos AS t
LEFT JOIN dim_cliente AS c
ON t.cliente_id = c.cliente_id
LEFT JOIN dim_producto AS p 
ON t.producto_id = p.producto_id
LEFT JOIN dim_fecha AS f 
ON t.fecha_pedido = f.fecha




Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

total_filas,detalles_distintos,lineas_sin_cliente,lineas_sin_producto,lineas_sin_fecha,importe_neto_total_clp
61647,61647,0,0,0,4666686966



#### 2.4.1.1 Resultados de validación del modelo dimensional

Se evaluó la integridad de las relaciones entre `fact_detalle_pedidos` y las dimensiones `dim_cliente`, `dim_producto` y `dim_fecha`.

Las relaciones se establecieron mediante las claves de cliente, producto y fecha de creación del pedido, utilizando `LEFT JOIN` para conservar todas las líneas de hechos y detectar referencias sin correspondencia.

##### Resultados

| Indicador | Resultado |
|---|---:|
| Total de líneas relacionadas | 61.647 |
| Identificadores de detalle distintos | 61.647 |
| Líneas sin cliente asociado | 0 |
| Líneas sin producto asociado | 0 |
| Líneas sin fecha asociada | 0 |
| Importe neto total (CLP) | $4.666.686.966 |

##### Hallazgos

Las 61.647 líneas de hechos encuentran correspondencia en las tres dimensiones implementadas.

No se detectaron duplicaciones de identificadores ni alteraciones en la cantidad de registros después de establecer las relaciones.

El importe neto agregado coincide con el valor de referencia de la auditoría inicial, confirmando que la integración dimensional no modifica el resultado monetario.

##### Conclusión

El modelo dimensional supera los controles de integridad referencial, conservación de granularidad y conciliación monetaria aplicados.

Las cuatro vistas conforman una estructura analítica consistente para desarrollar indicadores comerciales y preparar su posterior consumo desde herramientas de Business Intelligence.

Los importes comprenden los tres estados comerciales, por lo que su interpretación requiere aplicar las reglas de negocio correspondientes.

**Estado del control:** conforme.


In [95]:
%%sql

SELECT 
    c.segmento,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(t.costo_linea_clp) AS costo_entregado_clp,
    SUM(t.margen_bruto_estimado_linea_clp) AS margen_bruto_estimado_clp,
    ROUND(100.0 * SUM(t.margen_bruto_estimado_linea_clp) / NULLIF(SUM(t.importe_neto_linea_clp),0) ,2) AS pct_margen_bruto_estimado
FROM fact_detalle_pedidos AS t
LEFT JOIN dim_cliente AS c
ON c.cliente_id = t.cliente_id
WHERE t.estado = 'entregado'
GROUP BY c.segmento
ORDER BY importe_neto_entregado_clp DESC;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

segmento,pedidos_entregados,importe_neto_entregado_clp,costo_entregado_clp,margen_bruto_estimado_clp,pct_margen_bruto_estimado
Almacen,8295,2653879596,1898669592,755210004,28.46
Minimarket,3666,1154642886,826135098,328507788,28.45
Horeca,2773,625679552,447294596,178384956,28.51



### 2.4.2 Conciliación de indicadores comerciales por segmento

#### Objetivo

Verificar que el modelo dimensional permita calcular indicadores comerciales consistentes con los resultados de la auditoría original, incorporando costos y márgenes brutos estimados.

#### Metodología

Se relacionaron las vistas `fact_detalle_pedidos` y `dim_cliente` mediante `cliente_id`.

El análisis consideró exclusivamente pedidos en estado `entregado`, agrupando los resultados por segmento comercial.

Se calcularon cantidades de pedidos distintos, importes netos, costos asociados y márgenes brutos estimados.

#### Resultados

| Segmento | Pedidos | Importe neto (CLP) | Costo (CLP) | Margen estimado (CLP) | Margen % |
|---|---:|---:|---:|---:|---:|
| Almacen | 8.295 | 2.653.879.596 | 1.898.669.592 | 755.210.004 | 28,46 % |
| Minimarket | 3.666 | 1.154.642.886 | 826.135.098 | 328.507.788 | 28,45 % |
| Horeca | 2.773 | 625.679.552 | 447.294.596 | 178.384.956 | 28,51 % |
| **Total** | **14.734** | **4.434.202.034** | **3.172.099.286** | **1.262.102.748** | **28,46 %** |

#### Hallazgos

Los pedidos entregados y los importes netos por segmento coinciden con los valores previamente conciliados.

Almacen registra el mayor importe neto y margen bruto estimado en términos absolutos.

Horeca presenta el mayor porcentaje de margen bruto estimado, aunque las diferencias entre segmentos son reducidas.

#### Consideraciones

Los márgenes corresponden a estimaciones calculadas a partir de precios netos y costos unitarios registrados en las líneas de pedido.

No incorporan gastos operacionales, financieros, tributarios adicionales ni otros elementos necesarios para determinar la utilidad neta.

Los importes de pedidos entregados no equivalen necesariamente a facturación emitida o cobros realizados.

#### Conclusión

El modelo dimensional reproduce los principales indicadores comerciales de referencia y permite incorporar medidas de costos y márgenes sin alterar la granularidad de los registros.

**Estado del control:** conforme.


In [96]:
%%sql

SELECT
    SUM(importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(costo_linea_clp) AS costo_entregado_clp,
    SUM(margen_bruto_estimado_linea_clp) AS margen_bruto_estimado_clp,
    ROUND(
        100.0 * SUM(margen_bruto_estimado_linea_clp)
        / NULLIF(SUM(importe_neto_linea_clp), 0),
        2
    ) AS pct_margen_bruto_estimado
FROM fact_detalle_pedidos
WHERE estado = 'entregado';

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

importe_neto_entregado_clp,costo_entregado_clp,margen_bruto_estimado_clp,pct_margen_bruto_estimado
4434202034,3172099286,1262102748,28.46



### 2.4.3 Conciliación general de medidas comerciales

#### Objetivo

Comprobar que las medidas monetarias calculadas directamente desde `fact_detalle_pedidos` coincidan con los resultados previamente obtenidos por segmento comercial.

#### Resultados

| Indicador | Resultado |
|---|---:|
| Importe neto entregado (CLP) | 4.434.202.034 |
| Costo asociado (CLP) | 3.172.099.286 |
| Margen bruto estimado (CLP) | 1.262.102.748 |
| Margen bruto estimado (%) | 28,46 % |

#### Hallazgos

Los importes y márgenes agregados coinciden con los valores consolidados de los segmentos Almacen, Minimarket y Horeca.

Se verifica la consistencia aritmética entre el importe neto, el costo asociado y el margen bruto estimado.

#### Conclusión

La conciliación confirma que las medidas definidas a nivel de línea pueden agregarse correctamente para obtener indicadores comerciales generales y por segmento.

**Estado del control:** conforme.

---

## 2.5 Diccionario de indicadores y reglas de negocio

### 2.5.1 Definiciones de indicadores

| Indicador | Definición | Unidad |
|---|---|---|
| Pedidos entregados | Cantidad de identificadores de pedido distintos con estado `entregado` | Pedidos |
| Unidades entregadas | Suma de `cantidad` de líneas asociadas a pedidos entregados | Unidades |
| Importe neto entregado | Suma de `importe_neto_linea_clp` para pedidos entregados | CLP |
| Costo asociado a entregas | Suma de `costo_linea_clp` para pedidos entregados | CLP |
| Margen bruto estimado | Diferencia entre importe neto entregado y costo asociado | CLP |
| Margen bruto estimado (%) | Margen bruto estimado dividido por importe neto entregado, multiplicado por 100 | % |

### 2.5.2 Reglas de negocio

1. **Estado comercial:** los indicadores de desempeño comercial entregado considerarán exclusivamente pedidos con estado `entregado`, salvo indicación contraria.

2. **Granularidad:** los importes, costos y cantidades se calcularán desde las líneas de pedido. Los pedidos se contabilizarán utilizando identificadores distintos para evitar duplicaciones.

3. **Importes monetarios:** los valores netos registrados en los pedidos no se interpretarán automáticamente como ingresos facturados o efectivamente cobrados.

4. **Rentabilidad:** el margen bruto estimado corresponde a la diferencia entre el importe neto registrado y el costo asociado. No representa utilidad operacional ni neta.

5. **Porcentajes:** los indicadores porcentuales se calcularán a partir de importes agregados, evitando promediar directamente porcentajes de diferentes grupos.

6. **Temporalidad:** la dimensión Fecha se relacionará inicialmente mediante `fecha_pedido`. Los análisis por fecha de entrega o vencimiento requerirán definiciones adicionales.

7. **Atributos descriptivos:** las dimensiones Cliente y Producto reflejan la información disponible en las tablas de origen. No se implementó historización de cambios en esta etapa.

### 2.5.3 Limitaciones

El modelo utiliza datos ficticios y se desarrolla con fines demostrativos.

Las medidas comerciales no incorporan necesariamente facturación, cobros, devoluciones, gastos operacionales o ajustes contables.

Las definiciones establecidas deberán respetarse en los análisis posteriores para mantener la consistencia entre SQL y Power BI.


In [97]:

from pathlib import Path
from contextlib import closing
import sqlite3

origen = Path("../data/distribuidora_cordillera.sqlite").resolve()
carpeta_sql = Path("../sql").resolve()

scripts = [
    "01_fact_detalle_pedidos.sql",
    "02_dim_cliente.sql",
    "03_dim_producto.sql",
    "04_dim_fecha.sql"
]

with closing(sqlite3.connect(origen.as_uri() + "?mode=ro", uri=True)) as db_origen, \
     closing(sqlite3.connect(":memory:")) as db_prueba:

    # Copia temporal de la base original en memoria
    db_origen.backup(db_prueba)

    # Reconstrucción de las vistas
    for archivo in scripts:
        sql = (carpeta_sql / archivo).read_text(encoding="utf-8")
        db_prueba.executescript(sql)
        print(f"OK: {archivo}")

    # Verificación de las vistas reconstruidas
    vistas = [
        "fact_detalle_pedidos",
        "dim_cliente",
        "dim_producto",
        "dim_fecha"
    ]

    print("\nRegistros por vista:")
    for vista in vistas:
        total = db_prueba.execute(
            f"SELECT COUNT(*) FROM {vista}"
        ).fetchone()[0]
        print(f"{vista}: {total:,}")

    neto_entregado = db_prueba.execute("""
        SELECT SUM(importe_neto_linea_clp)
        FROM fact_detalle_pedidos
        WHERE estado = 'entregado'
    """).fetchone()[0]

    print(f"\nImporte neto entregado: ${neto_entregado:,}")


OK: 01_fact_detalle_pedidos.sql
OK: 02_dim_cliente.sql
OK: 03_dim_producto.sql
OK: 04_dim_fecha.sql

Registros por vista:
fact_detalle_pedidos: 61,647
dim_cliente: 600
dim_producto: 180
dim_fecha: 1,004

Importe neto entregado: $4,434,202,034



## 2.6 Reproducibilidad del modelo analítico

### Objetivo

Verificar que las vistas del modelo dimensional puedan reconstruirse utilizando exclusivamente la base de datos operacional original y sus definiciones SQL independientes.

### Metodología

Se utilizó una conexión de solo lectura a `distribuidora_cordillera.sqlite` para generar una copia temporal de la base de datos en memoria.

Sobre esta copia se ejecutaron secuencialmente los cuatro scripts SQL almacenados en el directorio `sql/`:

1. `01_fact_detalle_pedidos.sql`
2. `02_dim_cliente.sql`
3. `03_dim_producto.sql`
4. `04_dim_fecha.sql`

Posteriormente, se comprobó la cantidad de registros de cada vista y se concilió el importe neto correspondiente a pedidos entregados.

### Resultados

| Componente | Registros |
|---|---:|
| `fact_detalle_pedidos` | 61.647 |
| `dim_cliente` | 600 |
| `dim_producto` | 180 |
| `dim_fecha` | 1.004 |

**Importe neto entregado reconstruido:** $4.434.202.034 CLP.

### Hallazgos

Los cuatro scripts se ejecutaron correctamente y permitieron reconstruir las vistas del modelo dimensional.

Las cantidades de registros coinciden con los resultados de las validaciones anteriores.

El importe neto entregado coincide con el valor de referencia obtenido durante la auditoría y conciliación comercial.

La prueba se realizó sin modificar la base de datos original ni conservar archivos temporales adicionales.

### Conclusión

El modelo analítico supera la prueba de reconstrucción ejecutada, demostrando que sus vistas pueden implementarse a partir de definiciones SQL independientes.

Esto facilita la trazabilidad, el mantenimiento y la reutilización del modelo en etapas posteriores.

**Estado del control:** conforme.

---

## 2.7 Conclusiones del modelado dimensional

### Resumen del trabajo desarrollado

Se diseñó e implementó una capa analítica comercial basada en un esquema estrella lógico, utilizando SQLite y vistas SQL.

El modelo integra una tabla de hechos a nivel de línea de pedido y tres dimensiones principales: Cliente, Producto y Fecha.

La implementación permite analizar cantidades, pedidos, importes netos, costos y márgenes brutos estimados mediante atributos comerciales y temporales.

### Resultados principales

- **61.647 líneas de pedido** integradas en la tabla de hechos.
- **600 clientes** representados en la dimensión Cliente.
- **180 productos** enriquecidos con sus categorías.
- **1.004 fechas** disponibles para el análisis temporal.
- **0 líneas sin correspondencia** con las dimensiones evaluadas.
- **$4.434.202.034 CLP** de importe neto asociado a pedidos entregados.
- **$1.262.102.748 CLP** de margen bruto estimado asociado a pedidos entregados.
- **28,46 %** de margen bruto estimado agregado.

### Consideraciones y limitaciones

El modelo utiliza datos ficticios y tiene fines de aprendizaje y demostración profesional.

Los importes de pedidos entregados no representan necesariamente facturación emitida ni ingresos efectivamente cobrados.

El margen bruto estimado no incorpora todos los componentes necesarios para determinar la rentabilidad operacional o neta.

Las dimensiones Cliente y Producto no mantienen un historial de modificaciones de sus atributos descriptivos.

La dimensión Fecha se relaciona inicialmente con la fecha de creación de los pedidos. Otros roles temporales requerirán tratamiento adicional.

Las relaciones del esquema estrella fueron verificadas mediante SQL; su implementación en un modelo semántico de Power BI corresponde a una etapa posterior.

### Próximas etapas

El modelo construido constituirá la base para:

1. Análisis comercial avanzado mediante SQL.
2. Evaluación de evolución temporal, segmentación y concentración comercial.
3. Análisis de rentabilidad bruta estimada por cliente, producto y categoría.
4. Implementación de un modelo semántico e indicadores DAX en Power BI.
5. Desarrollo de visualizaciones y recomendaciones basadas en los resultados.

### Conclusión general

El proyecto dispone de una capa dimensional comercial implementada, documentada, validada y reproducible.

Las comprobaciones realizadas permiten utilizar las vistas como punto de partida para desarrollar análisis comerciales posteriores, manteniendo criterios consistentes de granularidad, integridad y cálculo de indicadores.

**Estado del Notebook 02:** finalizado.
